# 01 — Data cleaning

**Goal:** combine NASA POWER (temperature, humidity), CHIRPS (rainfall) and ERA5 (temperature cross-check) into one clean daily dataset, ready for analysis.

**Input:** `data/raw/*.csv` (NASA POWER), `data/raw/chirps/*.csv` (CHIRPS) and `data/raw/era5/*.csv` (ERA5), one file per city

**Output:** `data/processed/madagascar_climate_daily.csv`

In [1]:
from pathlib import Path

import pandas as pd

RAW_DIR= Path("../data/raw")
PROCESSED_DIR= Path("../data/processed")

## 1. Load and combine the raw files
Each city has its own file. We stack them into a single table and add the climate zone of each city.

In [2]:
ZONES= {
    "antananarivo" : "Central Highlands",
    "fianarantsoa" : "Southern Highlands",
    "toamasina" : "East coast",
    "mahajanga" : "West coast",
    "antsiranana" : "Far north",
    "toliara" : "Semi-arid south",
}

frames = [pd.read_csv(path, dtype={"date" : str}) for path in sorted(RAW_DIR.glob("*.csv"))]
df = pd.concat(frames, ignore_index = True)
df["zone"] = df["city"].map(ZONES)

print(df.shape)

(98616, 8)


In [3]:
print(df["city"].value_counts())
print("Cities without a zone: ", df["zone"].isna().sum())

city
antananarivo    16436
antsiranana     16436
fianarantsoa    16436
mahajanga       16436
toamasina       16436
toliara         16436
Name: count, dtype: int64
Cities without a zone:  0


### 1b. Add CHIRPS rainfall
NASA POWER rainfall proved unreliable in amount (see the validation section below), so daily rainfall from CHIRPS is added as a separate column, `rain_chirps`, and used for the analysis.

In [4]:
CHIRPS_DIR = RAW_DIR / "chirps"

chirps = pd.concat(
    [pd.read_csv(path, dtype={"date": str}) for path in sorted(CHIRPS_DIR.glob("*.csv"))],
    ignore_index=True,
)
df = df.merge(chirps, on=["city", "date"], how="left", validate="one_to_one")

print(df.shape)
print("Days without CHIRPS value:", df["rain_chirps"].isna().sum())

(98616, 9)
Days without CHIRPS value: 0


### 1c. Add ERA5 temperatures (cross-check)
NASA POWER shows a suspicious warming trend in Fianarantsoa. Daily temperatures from the ERA5 reanalysis (Open-Meteo archive, `models=era5`, 0.25° grid) are added as separate columns to cross-check the temperature trends with an independent source.

In [5]:
ERA5_DIR = RAW_DIR / "era5"

era5 = pd.concat(
    [pd.read_csv(path, dtype={"date": str}) for path in sorted(ERA5_DIR.glob("*.csv"))],
    ignore_index=True,
)
df = df.merge(era5, on=["city", "date"], how="left", validate="one_to_one")

print(df.shape)
print("Days without ERA5 value:", df["tmean_era5"].isna().sum())

(98616, 12)
Days without ERA5 value: 0


## 2. Convert dates and check data types
Dates arrive as text (`19810101`). We convert them to real dates so we can extract years, months and seasons later.

In [6]:
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 98616 entries, 0 to 98615
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         98616 non-null  datetime64[us]
 1   city         98616 non-null  str           
 2   T2M          98616 non-null  float64       
 3   T2M_MAX      98616 non-null  float64       
 4   T2M_MIN      98616 non-null  float64       
 5   PRECTOTCORR  98616 non-null  float64       
 6   RH2M         98616 non-null  float64       
 7   zone         98616 non-null  str           
 8   rain_chirps  98616 non-null  float64       
 9   tmean_era5   98616 non-null  float64       
 10  tmax_era5    98616 non-null  float64       
 11  tmin_era5    98616 non-null  float64       
dtypes: datetime64[us](1), float64(9), str(2)
memory usage: 9.0 MB


## 3. Missing values
NASA POWER marks missing data with `-999` and CHIRPS with `-9999`. We replace both codes with a real missing value (`NaN`) so they can never distort averages, then count what is missing.

In [7]:
VALUE_COLS = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "rain_chirps",
              "tmean_era5", "tmax_era5", "tmin_era5"]

n_fill = df[VALUE_COLS].isin([-999, -9999]).sum().sum()
df[VALUE_COLS] = df[VALUE_COLS].replace([-999, -9999], float("nan"))

print("Missing-value codes replaced:", n_fill)
print(df[VALUE_COLS].isna().sum())

Missing-value codes replaced: 0
T2M            0
T2M_MAX        0
T2M_MIN        0
PRECTOTCORR    0
RH2M           0
rain_chirps    0
tmean_era5     0
tmax_era5      0
tmin_era5      0
dtype: int64


## 4. Consistency checks
Each check counts the rows that break a rule that should always hold. All counts should be 0.

In [8]:
full_period = pd.date_range("1981-01-01", "2025-12-31", freq="D")

checks = {
    "Duplicate (city, date) rows": df.duplicated(["city", "date"]).sum(),
    "Missing days":                sum(len(full_period.difference(g["date"])) for _, g in df.groupby("city")),
    "T2M_MIN above T2M":           (df["T2M_MIN"] > df["T2M"]).sum(),
    "T2M above T2M_MAX":           (df["T2M"] > df["T2M_MAX"]).sum(),
    "Negative rainfall":           (df["PRECTOTCORR"] < 0).sum(),
    "Humidity outside 0–100 %":    (~df["RH2M"].between(0, 100)).sum(),
    "Temperature outside -5–50 °C": ((df["T2M_MIN"] < -5) | (df["T2M_MAX"] > 50)).sum(),
    "Negative CHIRPS rainfall":    (df["rain_chirps"] < 0).sum(),
    "ERA5: min above mean or mean above max": ((df["tmin_era5"] > df["tmean_era5"]) | (df["tmean_era5"] > df["tmax_era5"])).sum(),
    "ERA5: temperature outside -5-50 °C":      ((df["tmin_era5"] < -5) | (df["tmax_era5"] > 50)).sum(),
}
pd.Series(checks, name="Rows failing")

Duplicate (city, date) rows               0
Missing days                              0
T2M_MIN above T2M                         0
T2M above T2M_MAX                         0
Negative rainfall                         0
Humidity outside 0–100 %                  0
Temperature outside -5–50 °C              0
Negative CHIRPS rainfall                  0
ERA5: min above mean or mean above max    0
ERA5: temperature outside -5-50 °C        0
Name: Rows failing, dtype: int64

In [9]:
df[VALUE_COLS].describe().round(1)

,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,rain_chirps,tmean_era5,tmax_era5,tmin_era5
count,98616.0,98616.0,98616.0,98616.0,98616.0,98616.0,98616.0,98616.0,98616.0
mean,22.5,26.3,19.4,4.3,78.4,4.0,23.3,28.0,19.6
std,4.6,4.1,5.3,8.1,8.9,11.4,4.2,4.1,4.6
min,7.6,11.3,3.0,0.0,43.9,0.0,9.8,12.7,3.0
25%,19.4,23.6,15.5,0.2,72.6,0.0,20.6,25.3,16.8
50%,23.5,27.1,20.7,1.2,80.2,0.0,24.2,28.6,20.4
75%,26.1,29.1,23.8,4.8,85.4,0.0,26.7,31.2,23.6
max,33.5,39.8,29.2,244.1,95.5,433.9,33.5,39.5,29.4


In [10]:
for col, how in [("T2M_MAX", "idxmax"), ("T2M_MIN", "idxmin"), ("PRECTOTCORR", "idxmax"), ("rain_chirps", "idxmax")]:
    row = df.loc[getattr(df[col], how)()]
    print(f"{col:12} {row[col]:>6}  {row['city']:13} {row['date'].date()}")

T2M_MAX       39.84  toliara       2025-01-11
T2M_MIN        2.95  fianarantsoa  2009-06-30
PRECTOTCORR  244.12  antananarivo  2020-04-12
rain_chirps  433.9154968261719  toamasina     1992-04-07


**Result:** all checks pass. No missing values, no duplicates, no missing days and no impossible values were found, so no rows were removed or modified.

In [11]:
mask = df["date"].between("2020-04-08", "2020-04-16")
df[mask].pivot(index="date", columns="city", values="PRECTOTCORR").round(1)

city,antananarivo,antsiranana,fianarantsoa,mahajanga,toamasina,toliara
date,,,,,,
2020-04-08,6.8,0.4,11.0,0.2,1.7,0.0
2020-04-09,3.8,5.2,10.0,0.1,5.2,0.1
2020-04-10,1.1,4.4,17.3,0.7,8.8,0.0
2020-04-11,50.0,0.7,9.9,29.2,64.8,0.0
2020-04-12,244.1,5.3,14.2,20.0,65.6,0.2
2020-04-13,95.4,5.6,6.3,55.8,83.3,0.0
2020-04-14,107.9,6.3,7.5,2.0,63.6,0.0
2020-04-15,147.0,2.6,8.9,4.6,82.3,0.2
2020-04-16,33.6,4.0,3.7,2.7,18.8,0.3


In [12]:
mask = df["date"].between("2020-04-08", "2020-04-16")
df[mask & (df["city"] == "antananarivo")].set_index("date")[["PRECTOTCORR", "rain_chirps"]].round(1)

,PRECTOTCORR,rain_chirps
date,,
2020-04-08,6.8,0.0
2020-04-09,3.8,0.0
2020-04-10,1.1,0.0
2020-04-11,50.0,0.0
2020-04-12,244.1,0.0
2020-04-13,95.4,0.0
2020-04-14,107.9,0.0
2020-04-15,147.0,0.0
2020-04-16,33.6,0.0


**Note on the NASA POWER rainfall record:** NASA POWER shows a six-day wet episode in Antananarivo (about 680 mm, 11–16 April 2020), but CHIRPS records 0 mm on the same days and no matching report was found. The episode is most likely a model artefact. It is kept in the raw data, and NASA POWER rainfall is not used for the analysis.

## 5. Add time columns
- `year`, `month`: extracted from the date.
- `season`: **rainy** (November–April) or **dry** (May–October).
- `hydro_year`: a "hydrological year" running from 1 July to 30 June, labelled by its starting year. The rainy season (November–April) crosses New Year; with `hydro_year`, each rainy season belongs to a single year instead of being split in two.

In [13]:
RAINY_MONTHS = [11, 12, 1, 2, 3, 4]

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["season"] = df["month"].isin(RAINY_MONTHS).map({True: "rainy", False: "dry"})
df["hydro_year"] = df["year"].where(df["month"] >= 7, df["year"] - 1)

df.groupby("hydro_year").size().iloc[[0, 1, -2, -1]]

hydro_year
1980    1086
1981    2190
2024    2190
2025    1104
dtype: int64

## 5b. Rainfall validation
Mean annual rainfall from NASA POWER and CHIRPS, compared with published long-term normals (mostly station records, different periods: use for order of magnitude only).

In [14]:
REFERENCE_MM = {
    "toamasina": 3368,     # NOAA station normals 1961–1990
    "antananarivo": 1365,
    "mahajanga": 1476,     # NOAA station normals 1961–1990
    "fianarantsoa": 1085,
    "toliara": 360,        # semi-arid south-west, regional average
    "antsiranana": 940,
}

annual = df.groupby(["city", "year"])[["PRECTOTCORR", "rain_chirps"]].sum().groupby("city").mean()
annual["reference"] = pd.Series(REFERENCE_MM)
annual["nasa_error_%"] = (annual["PRECTOTCORR"] / annual["reference"] - 1) * 100
annual["chirps_error_%"] = (annual["rain_chirps"] / annual["reference"] - 1) * 100
annual.sort_values("reference").round(0)

,PRECTOTCORR,rain_chirps,reference,nasa_error_%,chirps_error_%
city,,,,,
toliara,550.0,312.0,360,53.0,-13.0
antsiranana,1102.0,1156.0,940,17.0,23.0
fianarantsoa,2038.0,1151.0,1085,88.0,6.0
antananarivo,2371.0,1322.0,1365,74.0,-3.0
mahajanga,1417.0,1470.0,1476,-4.0,-0.0
toamasina,1926.0,3271.0,3368,-43.0,-3.0


## 6. Save the clean dataset
Columns are reordered (identifiers first, then measurements) and rows are sorted by city and date. The raw files in `data/raw/` are never modified.

In [15]:
COLUMN_ORDER = ["date", "city", "zone", "year", "month", "season", "hydro_year", *VALUE_COLS]

df = df[COLUMN_ORDER].sort_values(["city", "date"]).reset_index(drop=True)

out_path = PROCESSED_DIR / "madagascar_climate_daily.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df):,} rows × {df.shape[1]} columns to {out_path}")

Saved 98,616 rows × 16 columns to ../data/processed/madagascar_climate_daily.csv


In [16]:
check = pd.read_csv(out_path, parse_dates=["date"])
print(check.shape)
check.head(3)

(98616, 16)


,date,city,zone,year,month,season,hydro_year,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,rain_chirps,tmean_era5,tmax_era5,tmin_era5
0,1981-01-01,antananarivo,Central Highlands,1981,1,rainy,1980,19.37,23.98,15.26,0.45,84.79,15.564335,20.6,27.0,17.1
1,1981-01-02,antananarivo,Central Highlands,1981,1,rainy,1980,19.40,23.85,15.71,0.00,86.34,0.000000,21.5,26.6,17.9
2,1981-01-03,antananarivo,Central Highlands,1981,1,rainy,1980,18.74,24.08,14.82,0.00,84.12,0.000000,21.9,27.5,17.9
